In [ ]:
pip install pydub

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import os
import json
import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from google.colab import drive
from tqdm import tqdm

# Audio segment creation
import math
from pydub import AudioSegment
import shutil

In [ ]:
# Connect Google Drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Dataset location
SOURCE_PATH = '/content/drive/MyDrive/songo/songo_datasets'

# Path to labels and processed data file, json format.
JSON_PATH = '/content/drive/MyDrive/songo/cnn_4/out/data.json'

# Path to saved model
MODEL_PATH = '/content/drive/MyDrive/songo/cnn_4/model/mels_cnn_new.h5'

# Sampling rate.
sr = 22050

# Let's make sure all files have the same amount of samples, pick a duration right under 30 seconds.
TOTAL_SAMPLES = 30 * sr

# The dataset contains 999 files. Lets make it bigger.
# X amount of slices => X times more training examples.
NUM_SLICES = 10
SAMPLES_PER_SLICE = int(TOTAL_SAMPLES / NUM_SLICES)

# Genres to analyze
genres = ['deep_house', 'tech_house', 'melodic_techno', 'progressive', 'techno_peak_time', 'hard_techno', 'minimal', 'trance']

In [ ]:
# Create audio segments

def create_segments():
  for genre in genres: # Iterate over genres
      for index, file_name in enumerate(os.listdir(SOURCE_PATH + '/' + genre)): # Iterate over each file of a specific genre
        print('analyzing: ' + str(index) + '- ' + file_name + ' ')
        song_name = SOURCE_PATH + '/' + genre + '/' + file_name
        track = AudioSegment.from_file(song_name, 'aiff')
        duration = math.trunc(track.duration_seconds * 1000)
        ranges = map(lambda interval_number: ((duration/5)*interval_number - 15000, (duration/5)*interval_number + 15000) , range(1, 5))
        for range_idx, r in enumerate(ranges):
          sample_name = str(index + 1) + '-s' + str(range_idx + 1) + '.aiff'
          splitted = track[r[0]:r[1]]
          splitted.export(sample_name, format='aiff')
          shutil.move(sample_name, SOURCE_PATH + '/' + genre + '_data' + '/' + sample_name)
          print('created: ' + str(range_idx))

In [ ]:
def preprocess_data(source_path, json_path):

    # Let's create a dictionary of labels and processed data.
    mydict = {
        "labels": [],
        "mfcc": []
        }

    # Let's browse each file, slice it and generate the 13 band mfcc for each slice.
    for i, genre in enumerate(genres):
        genre_folder = genre + '_data'
        for file_name in tqdm(os.listdir(os.path.join(SOURCE_PATH, genre_folder))):
            song, sr = librosa.load(os.path.join(SOURCE_PATH, genre_folder, file_name), duration=30)

            for s in range(NUM_SLICES):
                start_sample = SAMPLES_PER_SLICE * s
                end_sample = start_sample + SAMPLES_PER_SLICE
                mfcc = librosa.feature.mfcc(y=song[start_sample:end_sample], sr=sr, n_mfcc=13)
                mfcc = mfcc.T
                mydict["labels"].append(i)
                mydict["mfcc"].append(mfcc.tolist())

    # Let's write the dictionary in a json file.
    with open(json_path, 'w') as f:
        json.dump(mydict, f)
    f.close()

In [ ]:
def load_data(json_path):

    with open(json_path, 'r') as f:
        data = json.load(f)
    f.close()

    # Let's load our data into numpy arrays for TensorFlow compatibility.
    X = np.array(data["mfcc"])
    y = np.array(data["labels"])

    return X, y

In [ ]:
def prepare_datasets(inputs, targets, split_size):

    # Group inputs by segments of same track
    group_indexes = range(len(inputs) // 40)

    # Creating a validation set and a test set.
    group_indexes_train, group_indexes_val = train_test_split(group_indexes, test_size=split_size)
    group_indexes_train, group_indexes_test = train_test_split(group_indexes_train, test_size=split_size)

    print(group_indexes_train)
    print(group_indexes_val)
    print(group_indexes_test)

    # Create datasets
    inputs_train = []
    inputs_val = []
    inputs_test = []
    targets_train = []
    targets_val = []
    targets_test = []

    for i_tr in group_indexes_train:
      for j_tr in range(i_tr*40, i_tr*40+40):
        inputs_train.append(inputs[j_tr])
        targets_train.append(targets[j_tr])

    for i_te in group_indexes_test:
      for j_te in range(i_te*40, i_te*40+40):
        inputs_test.append(inputs[j_te])
        targets_test.append(targets[j_te])

    for i_val in group_indexes_val:
      for j_val in range(i_val*40, i_val*40+40):
        inputs_val.append(inputs[j_val])
        targets_val.append(targets[j_val])

    inputs_train = np.array(inputs_train)
    inputs_val = np.array(inputs_val)
    inputs_test = np.array(inputs_test)
    targets_train = np.array(targets_train)
    targets_val = np.array(targets_val)
    targets_test = np.array(targets_test)

    # Our CNN model expects 3D input shape.
    inputs_train = inputs_train[..., np.newaxis]
    inputs_val = inputs_val[..., np.newaxis]
    inputs_test = inputs_test[..., np.newaxis]

    return inputs_train, inputs_val, inputs_test, targets_train, targets_val, targets_test

In [ ]:
def design_model(input_shape):

    # Let's design the model architecture.
    model = tf.keras.models.Sequential([

        tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        tf.keras.layers.MaxPooling2D((3,3), strides=(2,2), padding='same'),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D((3,3), strides=(2,2), padding='same'),
        tf.keras.layers.BatchNormalization(),

        tf.keras.layers.Conv2D(32, (2,2), activation='relu'),
        tf.keras.layers.MaxPooling2D((3,3), strides=(2,2), padding='same'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(len(np.unique(targets)), activation='softmax')
    ])

    return model

In [ ]:
def make_prediction(model, X, y):
    predictions = model.predict(X)
    print(predictions[0:10])

    total_accuracy = 0

    for i in range(len(predictions) // 40):
      song_pred = np.zeros(len(genres))
      for j in range(i*40, (i+1)*40):
        for k in range(len(genres)):
            song_pred[k] += predictions[j][k]

      predicted_genre = np.argmax(song_pred)
      if(i==0):
        print(song_pred)
      true_genre = y[i*40]
      if predicted_genre == true_genre:
        total_accuracy += 1

    total_songs = len(predictions) // 40
    total_accuracy /= total_songs
    print("Total accuracy: " + str(total_accuracy))

In [ ]:
def plot_performance(hist):

    acc = hist.history['acc']
    val_acc = hist.history['val_acc']
    loss = hist.history['loss']
    val_loss = hist.history['val_loss']

    epochs = range(len(acc))

    plt.plot(epochs, acc, 'r', label='Training accuracy')
    plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
    plt.title('Training and validation accuracy')
    plt.legend()
    plt.figure()

    plt.plot(epochs, loss, 'r', label='Training Loss')
    plt.plot(epochs, val_loss, 'b', label='Validation Loss')
    plt.title('Training and validation loss')
    plt.legend()

    plt.show()

In [ ]:
preprocess_data(source_path=SOURCE_PATH, json_path=JSON_PATH)

In [ ]:
inputs, targets = load_data(json_path=JSON_PATH)

Xtrain, Xval, Xtest, ytrain, yval, ytest = prepare_datasets(inputs, targets, 0.2)

input_shape = (Xtrain.shape[1], Xtrain.shape[2], 1)

In [ ]:
model = design_model(input_shape)

# Selection of the optimizer, loss type and metrics for performance evaluation.
model.compile(optimizer = tf.keras.optimizers.RMSprop(lr=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics = ['acc']
                  )

model.summary()

# Training the model.
history = model.fit(Xtrain, ytrain,
                    validation_data=(Xval, yval),
                    epochs=30,
                    batch_size=32,
                    )

plot_performance(history)

# Testing the model on never seen before data.
make_prediction(model, Xtest, ytest)

In [ ]:
# Save model

model.save(MODEL_PATH)